In [ ]:

# ============================================================
# PROJECT 4 - SOCIAL MEDIA TREND INTELLIGENCE
# REAL SENTIMENT140 TWITTER DATA | ONE COLAB CELL
# ============================================================

import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "pandas", "numpy", "plotly", "requests"
])

import pandas as pd
import numpy as np
import plotly.express as px
import requests
import zipfile
import re
from collections import Counter
from pathlib import Path
from plotly.io import to_html
from IPython.display import display

# 1. SETTINGS
PROJECT = Path("/content/Social_Media_Trend_Intelligence")
PROJECT.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ZIP = False

DATA_URL = (
    "https://huggingface.co/datasets/"
    "andrea-t94/TwitterSentiment140/resolve/main/"
    "training.1600000.processed.noemoticon.csv"
)

RAW_FILE = Path("/content/sentiment140.csv")

# 2. DOWNLOAD DATASET ONCE
if not RAW_FILE.exists() or RAW_FILE.stat().st_size == 0:
    print("Downloading real Sentiment140 tweets...")

    with requests.get(
        DATA_URL, stream=True, timeout=120
    ) as response:
        response.raise_for_status()

        with open(RAW_FILE, "wb") as output:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    output.write(chunk)
else:
    print("Using previously downloaded dataset.")

# 3. READ REAL DATA
columns = [
    "Sentiment_Code", "Tweet_ID", "Date",
    "Query", "User", "Text"
]

df = pd.read_csv(
    RAW_FILE,
    header=None,
    names=columns,
    encoding="latin1",
    low_memory=False
)

print("Original records:", len(df))

# 4. CLEAN DATA
df["Sentiment_Code"] = pd.to_numeric(
    df["Sentiment_Code"], errors="coerce"
)

df = df[df["Sentiment_Code"].isin([0, 4])].copy()

df["Text"] = df["Text"].astype("string").str.strip()
df = df.dropna(subset=["Text"])
df = df[df["Text"].str.len() > 0]

df["Sentiment"] = df["Sentiment_Code"].map({
    0: "Negative",
    4: "Positive"
})

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%a %b %d %H:%M:%S %Z %Y",
    errors="coerce"
)

# Some pandas versions cannot parse the timezone
# abbreviation. Retry after removing it.
if df["Date"].notna().sum() == 0:
    original_dates = pd.read_csv(
        RAW_FILE,
        header=None,
        usecols=[2],
        encoding="latin1"
    ).iloc[:, 0]

    # Re-read the source date column safely.
    raw_dates = pd.read_csv(
        RAW_FILE,
        header=None,
        usecols=[2],
        encoding="latin1"
    ).iloc[:, 0]

    df["Date"] = pd.to_datetime(
        raw_dates.loc[df.index].astype(str)
        .str.replace(
            r"\s+[A-Z]{2,5}\s+(\d{4})$",
            r" \1",
            regex=True
        ),
        format="%a %b %d %H:%M:%S %Y",
        errors="coerce"
    )

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.date
df["Hour"] = df["Date"].dt.hour

df["Text_Length"] = df["Text"].str.len()
df["Word_Count"] = df["Text"].str.split().str.len()

df["Has_Hashtag"] = df["Text"].str.contains(
    r"#\w+", regex=True, na=False
)

df["Has_Mention"] = df["Text"].str.contains(
    r"@\w+", regex=True, na=False
)

df["Has_URL"] = df["Text"].str.contains(
    r"https?://|www\.", regex=True, na=False
)

df = df.reset_index(drop=True)

if df.empty:
    raise ValueError("No usable tweet records.")

df.to_csv(
    PROJECT / "cleaned_social_media_data.csv",
    index=False
)

print("\nCLEANED DATA")
display(df.head())

# 5. KPIs
kpis = {
    "Total Tweets": len(df),
    "Positive Tweets": int(
        (df["Sentiment"] == "Positive").sum()
    ),
    "Negative Tweets": int(
        (df["Sentiment"] == "Negative").sum()
    ),
    "Positive Share (%)": round(
        (df["Sentiment"] == "Positive").mean() * 100,
        2
    ),
    "Unique User Labels": df["User"].nunique(),
    "Average Tweet Length": round(
        df["Text_Length"].mean(), 2
    ),
    "Tweets With Hashtags": int(
        df["Has_Hashtag"].sum()
    ),
    "Tweets With Mentions": int(
        df["Has_Mention"].sum()
    )
}

kpi_df = pd.DataFrame(
    list(kpis.items()),
    columns=["KPI", "Value"]
)

print("\nSOCIAL MEDIA KPIs")
display(kpi_df)

kpi_df.to_csv(
    PROJECT / "kpi_summary.csv",
    index=False
)

# 6. CHART FUNCTIONS
figures = []

def chart(fig, title):
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=500,
        margin=dict(l=45, r=40, t=80, b=55)
    )
    figures.append((title, fig))
    fig.show()

# CHART 1 - SENTIMENT DISTRIBUTION
sentiment_counts = (
    df["Sentiment"].value_counts()
    .rename_axis("Sentiment")
    .reset_index(name="Tweets")
)

chart(
    px.bar(
        sentiment_counts,
        x="Sentiment",
        y="Tweets",
        color="Sentiment"
    ),
    "1. Positive vs Negative Tweets"
)

# CHART 2 - SENTIMENT SHARE
chart(
    px.pie(
        sentiment_counts,
        names="Sentiment",
        values="Tweets",
        hole=0.45
    ),
    "2. Tweet Sentiment Share"
)

# CHART 3 - DAILY TWEET VOLUME
dated = df.dropna(subset=["Date"]).copy()

if not dated.empty:
    daily = (
        dated.groupby("Day")
        .size()
        .reset_index(name="Tweets")
    )

    daily["Day"] = pd.to_datetime(daily["Day"])

    chart(
        px.line(
            daily,
            x="Day",
            y="Tweets",
            markers=True
        ),
        "3. Daily Tweet Volume"
    )

# CHART 4 - DAILY SENTIMENT
if not dated.empty:
    daily_sentiment = (
        dated.groupby(["Day", "Sentiment"])
        .size()
        .reset_index(name="Tweets")
    )

    daily_sentiment["Day"] = pd.to_datetime(
        daily_sentiment["Day"]
    )

    chart(
        px.line(
            daily_sentiment,
            x="Day",
            y="Tweets",
            color="Sentiment"
        ),
        "4. Daily Sentiment Trend"
    )

# CHART 5 - HOURLY ACTIVITY
if not dated.empty:
    hourly = (
        dated.groupby("Hour")
        .size()
        .reset_index(name="Tweets")
    )

    chart(
        px.bar(
            hourly,
            x="Hour",
            y="Tweets",
            color="Tweets"
        ),
        "5. Tweet Activity by Hour"
    )

# CHART 6 - TWEET LENGTH
sample = df.sample(
    min(len(df), 50000),
    random_state=42
)

chart(
    px.histogram(
        sample,
        x="Text_Length",
        color="Sentiment",
        nbins=50,
        barmode="overlay"
    ),
    "6. Tweet Text Length Distribution (Sample)"
)

# CHART 7 - TOP HASHTAGS
hashtags = Counter()

for text in df.loc[
    df["Has_Hashtag"], "Text"
].dropna():
    hashtags.update(
        tag.lower()
        for tag in re.findall(r"#\w+", str(text))
    )

top_hashtags = pd.DataFrame(
    hashtags.most_common(20),
    columns=["Hashtag", "Occurrences"]
)

if not top_hashtags.empty:
    chart(
        px.bar(
            top_hashtags.sort_values("Occurrences"),
            x="Occurrences",
            y="Hashtag",
            orientation="h",
            color="Occurrences"
        ),
        "7. Top 20 Hashtags"
    )

# CHART 8 - MOST COMMON WORDS
stopwords = set("""
the a an and or but to of in on for with is are
was were be been being i you he she it we they
my your his her our their this that these those
at by from as so if not do did does have has
had will would can could should just im its
me him them us rt http https www com
""".split())

words = Counter()

for text in df["Text"].dropna():
    cleaned = re.sub(
        r"https?://\S+|www\.\S+|@\w+",
        " ",
        str(text).lower()
    )

    tokens = re.findall(r"[a-z]{3,}", cleaned)

    words.update(
        w for w in tokens
        if w not in stopwords
    )

top_words = pd.DataFrame(
    words.most_common(20),
    columns=["Word", "Occurrences"]
)

if not top_words.empty:
    chart(
        px.bar(
            top_words.sort_values("Occurrences"),
            x="Occurrences",
            y="Word",
            orientation="h",
            color="Occurrences"
        ),
        "8. Top 20 Most Frequent Words"
    )

# 7. CSV REPORTS
sentiment_counts.to_csv(
    PROJECT / "summary_sentiment.csv",
    index=False
)

top_hashtags.to_csv(
    PROJECT / "summary_hashtags.csv",
    index=False
)

top_words.to_csv(
    PROJECT / "summary_words.csv",
    index=False
)

if not dated.empty:
    daily_sentiment.to_csv(
        PROJECT / "summary_daily_sentiment.csv",
        index=False
    )

# 8. COMBINED HTML DASHBOARD
html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Social Media Trend Intelligence</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    margin: 25px;
}
h1, p { text-align: center; }
.chart {
    background: white;
    border-radius: 12px;
    padding: 15px;
    margin: 20px auto;
    max-width: 1300px;
}
</style>
</head>
<body>
<h1>SOCIAL MEDIA TREND INTELLIGENCE</h1>
<p>Historical Sentiment140 Twitter Data</p>
"""

for i, (title, fig) in enumerate(figures):
    html += (
        '<div class="chart">'
        + to_html(
            fig,
            full_html=False,
            include_plotlyjs=(i == 0)
        )
        + "</div>"
    )

html += "</body></html>"

dashboard_path = (
    PROJECT / "Social_Media_Dashboard.html"
)

dashboard_path.write_text(
    html, encoding="utf-8"
)

# 9. README
readme = f"""
PROJECT 4 - SOCIAL MEDIA TREND INTELLIGENCE

Source:
{DATA_URL}

Dataset:
Sentiment140

Records:
{len(df)}

Charts:
{len(figures)}

No synthetic posts generated.

Limitations:
Historical Twitter data, not live trends.
Labels are positive or negative only.
Sentiment140 uses distant supervision.
No verified likes, shares or impressions.
User labels are not unique-person counts.
"""

(PROJECT / "README.txt").write_text(
    readme, encoding="utf-8"
)

# 10. ZIP
zip_path = Path(
    "/content/Social_Media_Trend_Intelligence.zip"
)

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for file in PROJECT.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(
                    PROJECT.parent
                )
            )

print("\n" + "=" * 45)
print("PROJECT 4 COMPLETED")
print("=" * 45)
print("Clean tweets:", len(df))
print("Charts:", len(figures))
print("Dashboard:", dashboard_path)
print("ZIP:", zip_path)
print("Automatic download: OFF")

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
